# 07 · Driver skill + car pace (hierarchical Bayesian model)

For every observation of a driver's pace (% of a lap, lower = faster):

`pace = session offset + driver skill + car pace + noise`

**Observations**
- **Qualifying:** every lap each driver set in Q1, Q2 and Q3, as % behind that segment's fastest lap. Each segment gets its own offset, so track evolution isn't mistaken for pace.
- **Race:** each driver's pace relative to the field, from the per-race lap model (notebook 04's method), with its standard error added to the noise. A scale factor lets race gaps differ in size from one-lap gaps.

**Structure**
- **Driver skill:** one value per driver per season, drifting season to season; centred on each season's average driver.
- **Car pace:** one value per team per race, evolving race to race within a season (upgrades), starting fresh each season; centred on each race's average car.
- **Noise:** Student-t (heavy-tailed), separate for qualifying and race, so crashes or ruined laps don't drag estimates.

Teammates sharing a car separate skill from car pace; drivers who change teams link the grid together. Fitted with PyMC (NUTS). Takes about 6 minutes on Colab's CPU with the default settings.

**Limitation:** skill is measured against teammates. An unusually weak teammate, or a car built around one driver, inflates that driver's skill and deflates the car (e.g. Red Bull 2023).

Run notebooks 00, 03 and 05 first.

In [ ]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/YOUR_USERNAME/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

In [ ]:
# @title Install PyMC and the file format for saving fits (skipped if already installed)
import importlib.util
missing = [p for p in ("pymc", "h5netcdf", "h5py") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q {" ".join(missing)}
import pymc
print("PyMC", pymc.__version__)

In [ ]:
# @title Build the observations
WET_RULE = "tyres"  # @param ["tyres", "tyres_or_rain"]

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from f1sim.paths import get_paths
from f1sim.ingest import load_quali
from f1sim.ingest_results import load_results
from f1sim.ingest_tracing import load_race_ti
from f1sim.race_pace import clean_dry_laps, dry_races, driver_race_pace
from f1sim import skill_model as sm

paths = get_paths()
results = load_results(paths)
laps = load_race_ti("laps", paths)
race_pace, race_skipped = driver_race_pace(clean_dry_laps(laps, wet_rule=WET_RULE),
                                           expected_races=dry_races(laps, WET_RULE))
data = sm.prepare_data(load_quali(paths), race_pace, results)

names = (results.sort_values(["Year", "Round"]).groupby("DriverId")[["Abbreviation", "FullName"]]
         .last().reset_index())
team_names = (results.groupby(["TeamId", "Year"])["TeamName"]
              .agg(lambda s: s.value_counts().index[0]).reset_index())
print(f"Qualifying laps: {len(data.quali):,} | race-pace observations: {len(data.race):,} | "
      f"races: {len(data.races)} | driver-seasons: {len(data.skill_cells)} | "
      f"team-race car values: {len(data.car_cells):,}")
print(f"Race-pace rows that couldn't be matched to a Jolpica driver/team: {len(data.unmatched_race_rows)}")

In [ ]:
# @title Fit the model (or load the last saved fit)
FIT_NEW = True  # @param {type:"boolean"}
DRAWS = 1000  # @param {type:"integer"}
TUNE = 1000  # @param {type:"integer"}
CHAINS = 2  # @param {type:"integer"}

if FIT_NEW:
    idata = sm.fit(sm.build_model(data), draws=DRAWS, tune=TUNE, chains=CHAINS)
    saved = sm.save_fit(idata, data, paths)
    print("Saved to", saved["posterior"])
else:
    idata, data = sm.load_fit(paths)
    print("Loaded the saved fit")

diag = sm.diagnostics(idata)
print(f"Divergences: {diag['divergences']} | max R-hat: {diag['max_r_hat']:.3f} | "
      f"min bulk ESS: {diag['min_ess_bulk']:.0f}")
if diag["divergences"] > 0 or diag["max_r_hat"] > 1.01 or diag["min_ess_bulk"] < 200:
    print("Warning: sampling may not have converged. Try TUNE = 2000 or CHAINS = 4 and refit.")
diag["hyperparameters"]

In [ ]:
# @title Driver skill in one season (bars = 90% interval)
YEAR = 2025  # @param {type:"integer"}

skills = sm.skill_table(idata, data, names)
s = skills[skills["Year"] == YEAR].sort_values("Mean")
fig = go.Figure(go.Scatter(
    x=s["Mean"], y=s["Abbreviation"], mode="markers",
    error_x=dict(type="data", symmetric=False, array=s["High90"] - s["Mean"],
                 arrayminus=s["Mean"] - s["Low90"]),
    customdata=s[["FullName", "QualiEvents", "RaceObs"]],
    hovertemplate="%{customdata[0]}<br>skill %{x:.2f}%<br>quali events %{customdata[1]}, race obs %{customdata[2]}<extra></extra>"))
fig.add_vline(x=0, line_dash="dot")
fig.update_layout(title=f"{YEAR}: driver skill vs the season's average driver (% of lap; left = faster)",
                  yaxis={"categoryorder": "array", "categoryarray": s["Abbreviation"][::-1]},
                  height=650, xaxis_title="Skill (% of lap)")
fig.show()
s[["Abbreviation", "FullName", "Mean", "Low90", "High90", "QualiEvents", "RaceObs"]].round(3)

In [ ]:
# @title Car pace in one season (season average, bars = 90% interval)
cars = sm.car_season_table(idata, data, team_names)
c = cars[cars["Year"] == YEAR].sort_values("Mean")
fig = go.Figure(go.Scatter(
    x=c["Mean"], y=c["TeamName"], mode="markers",
    error_x=dict(type="data", symmetric=False, array=c["High90"] - c["Mean"],
                 arrayminus=c["Mean"] - c["Low90"])))
fig.add_vline(x=0, line_dash="dot")
fig.update_layout(title=f"{YEAR}: car pace vs the average car (% of lap; left = faster)",
                  yaxis={"categoryorder": "array", "categoryarray": c["TeamName"][::-1]},
                  xaxis_title="Car pace (% of lap)")
fig.show()
c[["TeamName", "Mean", "Low90", "High90", "Races"]].round(3)

In [ ]:
# @title Car pace race by race through the season (development and upgrades)
per_race = sm.car_table(idata, data, team_names)
pr = per_race[per_race["Year"] == YEAR]
fig = px.line(pr, x="Round", y="Mean", color="TeamName", markers=True, hover_data=["Event"],
              labels={"Mean": "Car pace (% of lap; lower = faster)"},
              title=f"{YEAR}: car pace by round")
fig.update_yaxes(autorange="reversed")
fig.show()

In [ ]:
# @title Drivers' skill across seasons
DRIVERS = "VER,HAM,ALO,LEC,NOR"  # @param {type:"string"}

wanted = [d.strip().upper() for d in DRIVERS.split(",")]
dd = skills[skills["Abbreviation"].isin(wanted)].copy()
dd["ErrHigh"], dd["ErrLow"] = dd["High90"] - dd["Mean"], dd["Mean"] - dd["Low90"]
fig = px.line(dd, x="Year", y="Mean", color="Abbreviation", markers=True,
              error_y="ErrHigh", error_y_minus="ErrLow",
              labels={"Mean": "Skill (% of lap; lower = faster)"},
              title="Driver skill by season (bars = 90% interval)")
fig.update_yaxes(autorange="reversed")
fig.show()